In [11]:
import pandas as pd
import numpy as np
from functools import reduce

from pymongo import MongoClient

In [2]:
def data_preprocessing(data, year, quarter):
    data['year'] = year
    data['quarter'] = quarter 
    data2 = data[data['VISA_CLASS']=='H-1B']

    return data2

In [12]:
def concat_common_columns(data_list):
    """
    Concatenates multiple DataFrames on their common columns.

    Parameters:
        data_list (list): List of pandas DataFrames.

    Returns:
        pd.DataFrame: Concatenated DataFrame with only common columns.
    """
    if not data_list:
        return pd.DataFrame()  # return empty DataFrame if list is empty

    # Step 1: Find common columns across all dataframes
    common_cols = reduce(lambda x, y: x.intersection(y.columns), data_list, data_list[0].columns)

    # Step 2: Subset each dataframe to those common columns and concatenate
    aligned_data = [df[common_cols] for df in data_list]
    return pd.concat(aligned_data, ignore_index=True)

In [30]:
def filter_null_data(data2, threshold):
    null_columns = []
    for i in data2.columns:
        if data2[i].isna().sum()>(threshold*data2.shape[0]):
            print(i)
            print(data2[i].isna().sum())
            null_columns.append(i)
            
    data2_filtered = data2[[i for i in data2.columns if i not in null_columns]]
    return data2_filtered, null_columns

In [5]:
def create_column_list(columns, entity):
    column_list = [i for i in columns if entity in i]
    return column_list

In [36]:
def create_grouped_dict(data, null_columns):
    present = lambda cols: [col for col in cols if col in data.columns and col not in null_columns]

    grouped_dict = {}

    grouped_dict['employment_details'] = (
        create_column_list(data.columns, '_EMPLOYMENT') +
        present(['AMENDED_PETITION', 'TOTAL_WORKER_POSITIONS', 'NAICS_CODE'])
    )

    employer_details = create_column_list(data.columns, 'EMPLOYER_')
    grouped_dict['employer_details'] = employer_details
    grouped_dict['employer_poc_details'] = create_column_list(data.columns, 'EMPLOYER_POC')

    grouped_dict['agent_details'] = (
        create_column_list(data.columns, 'AGENT_') +
        present(['LAWFIRM_NAME_BUSINESS_NAME', 'STATE_OF_HIGHEST_COURT', 'NAME_OF_HIGHEST_STATE_COURT'])
    )

    grouped_dict['worksite_details'] = (
        create_column_list(data.columns, 'WORKSITE_') +
        present(['SECONDARY_ENTITY', 'SECONDARY_ENTITY_BUSINESS_NAME'])
    )

    grouped_dict['wage_details'] = create_column_list(data.columns, 'WAGE')
    grouped_dict['pw_columns'] = create_column_list(data.columns, 'PW_') + present(['PREVAILING_WAGE'])
    grouped_dict['preparer'] = create_column_list(data.columns, 'PREPARER_')

    return grouped_dict

In [7]:
def row_to_nested_json(row, grouped_columns):
    """
    Convert a row to a nested dictionary using predefined grouped columns.

    Parameters:
        row (pd.Series): A row from the DataFrame.
        grouped_columns (dict): Keys are group names, values are lists of columns.

    Returns:
        dict: A nested dictionary suitable for MongoDB insertion.
    """
    data = {}
    used_cols = set()

    # Create nested dictionaries for each group
    for group, columns in grouped_columns.items():
        nested = {}
        for col in columns:
            if col in row:
                nested[col] = row[col] if pd.notnull(row[col]) else None
                used_cols.add(col)
        data[group] = nested

    # Add remaining top-level columns (non-grouped)
    for col in row.index:
        if col not in used_cols:
            data[col] = row[col] if pd.notnull(row[col]) else None

    return data

In [8]:
q1_2025 = pd.read_excel('LCA_Disclosure_Data_FY2025_Q1.xlsx', nrows=10000)
q1_2024 = pd.read_excel('LCA_Disclosure_Data_FY2024_Q1.xlsx', nrows=10000)
q2_2024 = pd.read_excel('LCA_Disclosure_Data_FY2024_Q2.xlsx', nrows=10000)
q3_2024 = pd.read_excel('LCA_Disclosure_Data_FY2024_Q3.xlsx', nrows=10000)
q4_2024 = pd.read_excel('LCA_Disclosure_Data_FY2024_Q4.xlsx', nrows=10000)

In [9]:
q1_2025_h1 = data_preprocessing(q1_2025, 2025, 'Q1')
q1_2024_h1 = data_preprocessing(q1_2024, 2025, 'Q1')
q2_2024_h1 = data_preprocessing(q2_2024, 2025, 'Q2')
q3_2024_h1 = data_preprocessing(q3_2024, 2025, 'Q3')
q4_2024_h1 = data_preprocessing(q4_2024, 2025, 'Q4')

In [18]:
data_list = [q1_2025_h1, q1_2024_h1, q2_2024_h1, q3_2024_h1, q4_2024_h1]
final_data = concat_common_columns(data_list)

In [31]:
final_data, null_columns = filter_null_data(final_data, 0.7)

In [32]:
final_data.columns

Index(['CASE_NUMBER', 'CASE_STATUS', 'RECEIVED_DATE', 'DECISION_DATE',
       'VISA_CLASS', 'JOB_TITLE', 'SOC_CODE', 'SOC_TITLE',
       'FULL_TIME_POSITION', 'BEGIN_DATE', 'END_DATE',
       'TOTAL_WORKER_POSITIONS', 'NEW_EMPLOYMENT', 'CONTINUED_EMPLOYMENT',
       'CHANGE_PREVIOUS_EMPLOYMENT', 'NEW_CONCURRENT_EMPLOYMENT',
       'CHANGE_EMPLOYER', 'AMENDED_PETITION', 'EMPLOYER_NAME',
       'EMPLOYER_ADDRESS1', 'EMPLOYER_ADDRESS2', 'EMPLOYER_CITY',
       'EMPLOYER_STATE', 'EMPLOYER_POSTAL_CODE', 'EMPLOYER_COUNTRY',
       'EMPLOYER_PHONE', 'EMPLOYER_FEIN', 'NAICS_CODE',
       'EMPLOYER_POC_LAST_NAME', 'EMPLOYER_POC_FIRST_NAME',
       'EMPLOYER_POC_JOB_TITLE', 'EMPLOYER_POC_ADDRESS1',
       'EMPLOYER_POC_ADDRESS2', 'EMPLOYER_POC_CITY', 'EMPLOYER_POC_STATE',
       'EMPLOYER_POC_POSTAL_CODE', 'EMPLOYER_POC_COUNTRY',
       'EMPLOYER_POC_PHONE', 'EMPLOYER_POC_EMAIL',
       'AGENT_REPRESENTING_EMPLOYER', 'AGENT_ATTORNEY_LAST_NAME',
       'AGENT_ATTORNEY_FIRST_NAME', 'AGENT_ATTORNEY

In [21]:
for i in final_data.columns:
    print(i)
    print(final_data[i].dtype)

CASE_NUMBER
object
CASE_STATUS
object
RECEIVED_DATE
datetime64[ns]
DECISION_DATE
datetime64[ns]
VISA_CLASS
object
JOB_TITLE
object
SOC_CODE
object
SOC_TITLE
object
FULL_TIME_POSITION
object
BEGIN_DATE
datetime64[ns]
END_DATE
datetime64[ns]
TOTAL_WORKER_POSITIONS
int64
NEW_EMPLOYMENT
int64
CONTINUED_EMPLOYMENT
int64
CHANGE_PREVIOUS_EMPLOYMENT
int64
NEW_CONCURRENT_EMPLOYMENT
int64
CHANGE_EMPLOYER
int64
AMENDED_PETITION
int64
EMPLOYER_NAME
object
EMPLOYER_ADDRESS1
object
EMPLOYER_ADDRESS2
object
EMPLOYER_CITY
object
EMPLOYER_STATE
object
EMPLOYER_POSTAL_CODE
object
EMPLOYER_COUNTRY
object
EMPLOYER_PHONE
int64
EMPLOYER_FEIN
object
NAICS_CODE
int64
EMPLOYER_POC_LAST_NAME
object
EMPLOYER_POC_FIRST_NAME
object
EMPLOYER_POC_JOB_TITLE
object
EMPLOYER_POC_ADDRESS1
object
EMPLOYER_POC_ADDRESS2
object
EMPLOYER_POC_CITY
object
EMPLOYER_POC_STATE
object
EMPLOYER_POC_POSTAL_CODE
object
EMPLOYER_POC_COUNTRY
object
EMPLOYER_POC_PHONE
int64
EMPLOYER_POC_EMAIL
object
AGENT_REPRESENTING_EMPLOYER
object
AG

In [23]:
final_data.describe()

,RECEIVED_DATE,DECISION_DATE,BEGIN_DATE,END_DATE,TOTAL_WORKER_POSITIONS,NEW_EMPLOYMENT,CONTINUED_EMPLOYMENT,CHANGE_PREVIOUS_EMPLOYMENT,NEW_CONCURRENT_EMPLOYMENT,CHANGE_EMPLOYER,...,EMPLOYER_PHONE,NAICS_CODE,EMPLOYER_POC_PHONE,AGENT_ATTORNEY_PHONE,WORKSITE_WORKERS,WAGE_RATE_OF_PAY_FROM,WAGE_RATE_OF_PAY_TO,PREVAILING_WAGE,TOTAL_WORKSITE_LOCATIONS,year
count,48666,48666,48666,48666,48666.000000,48666.000000,48666.000000,48666.000000,48666.000000,48666.000000,...,4.866600e+04,48666.000000,4.866600e+04,3.585800e+04,48666.000000,4.866600e+04,1.555800e+04,48666.000000,48666.000000,48666.0
mean,2024-05-31 11:30:29.959314176,2024-06-26 16:33:33.265935360,2024-08-16 00:04:51.160152832,2027-07-26 22:39:18.574774784,1.749106,0.560350,0.435951,0.174372,0.011815,0.252558,...,1.594326e+10,432854.626413,1.625212e+10,1.510014e+10,1.746435,1.215880e+05,1.554960e+05,104714.886792,1.483459,2025.0
min,2019-11-04 00:00:00,2023-12-20 00:00:00,2019-11-04 00:00:00,2020-10-21 00:00:00,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,3.345345e+08,-4445.000000,3.345345e+08,1.212425e+09,1.000000,1.050000e+01,1.200000e+01,8.000000,1.000000,2025.0
25%,2024-03-19 00:00:00,2024-03-26 00:00:00,2024-05-17 00:00:00,2027-04-14 00:00:00,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.312558e+10,335311.000000,1.312396e+10,1.312342e+10,1.000000,8.600000e+04,1.066630e+05,78478.000000,1.000000,2025.0
50%,2024-06-20 00:00:00,2024-06-27 00:00:00,2024-09-23 00:00:00,2027-09-19 00:00:00,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.508816e+10,541330.000000,1.512471e+10,1.469291e+10,1.000000,1.154000e+05,1.450000e+05,102877.000000,1.000000,2025.0
75%,2024-09-20 00:00:00,2024-09-27 00:00:00,2024-12-19 00:00:00,2027-12-16 00:00:00,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,...,1.732249e+10,541511.000000,1.713966e+10,1.703678e+10,1.000000,1.532000e+05,1.955000e+05,131848.000000,2.000000,2025.0
max,2024-12-31 00:00:00,2024-12-31 00:00:00,2025-07-01 00:00:00,2028-06-30 00:00:00,170.000000,170.000000,50.000000,50.000000,170.000000,30.000000,...,9.762188e+12,926130.000000,9.779818e+12,9.184722e+11,170.000000,1.400000e+07,1.580270e+07,474773.000000,10.000000,2025.0
std,NaN,NaN,NaN,NaN,5.391094,3.936304,1.520468,0.985503,0.781795,0.810123,...,6.408801e+10,202467.734714,7.629649e+10,5.784129e+09,5.378288,9.294864e+04,1.863192e+05,47955.669348,0.747216,0.0


In [37]:
grouped_dict = create_grouped_dict(final_data, null_columns)

In [38]:
# Convert the DataFrame to list of nested dictionaries
nested_docs = final_data.apply(lambda row: row_to_nested_json(row, grouped_dict), axis=1).tolist()

In [ ]:
# Add details according to 
client = MongoClient("mongodb://localhost:27017/")
db = client["h1b_data"]
collection = db["applications_q1_2025"]